# Chef Bot: PDF-Grounded Cooking Assistant

GitHub: https://github.com/nshivachev/AI-for-Developers

ChefBot is a notebook-based AI assistant that answers cooking questions from an uploaded cookbook PDF.

It supports two retrieval modes:
- CAG: direct context retrieval from the vector database
- RAG: tool-based retrieval with memory and conversation context

The assistant can return responses as:
- Text
- Image (generated from retrieved context)
- Speech (audio narration)

## How To Run
1. Run cells from top to bottom.
2. Complete the Setup section below.
3. Ingest the PDF into Chroma.
4. Use the interactive chat section at the bottom.

## Notes
- RAG is intentionally stricter and uses retrieved evidence, memory, and recent conversation context.
- If `verbose=True`, debug logs will show retrieval and token diagnostics when available.

## Setup

- Set `OPENAI_API_KEY`
- Upload to PDF in session storage

In [ ]:
!pip install -q \
  chromadb \
  openai \
  pypdf \
  IPython \
  pydantic \
  ipywidgets \
  opentelemetry-api==1.38.0 \
  opentelemetry-sdk==1.38.0 \
  opentelemetry-proto==1.38.0 \
  opentelemetry-exporter-otlp-proto-common==1.38.0

In [ ]:
import base64
import json
import re
import time
import sys
from collections import deque
from pathlib import Path
from typing import Callable, Literal
from uuid import uuid4
from ipywidgets import widgets

from pydantic import BaseModel, Field
from pypdf import PdfReader
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from IPython.display import display, HTML, Audio, Image, Javascript
from openai import OpenAI
from google.colab import userdata

## Config

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
PDF_FILE = Path("/content/The_auxiliary_cook_book.pdf")
PATH_TO_CHROMA_DB = "/content/chromadb"
COLLECTION_NAME = "content"

SHORT_TERM_MAX = 10
LONG_TERM_MAX = 500
LONG_MEM_FILE = Path("long_memory.json")

openai_client = OpenAI(api_key=OPENAI_API_KEY)

## Short-term & Long-term memory

In [ ]:
short_term_memory = deque(maxlen=SHORT_TERM_MAX)
long_term_memory = deque(maxlen=LONG_TERM_MAX)

In [ ]:
def load_long_term_memory():
    if LONG_MEM_FILE.exists():
        try:
            with open(LONG_MEM_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
            long_term_memory.clear()
            long_term_memory.extend(data[-LONG_TERM_MAX:])
        except (json.JSONDecodeError, OSError) as e:
            print(f"Warning: Could not load long-term memory: {e}")
    return long_term_memory

In [ ]:
def save_long_term_memory():
    try:
        with open(LONG_MEM_FILE, "w", encoding="utf-8") as f:
            json.dump(list(long_term_memory), f, indent=2, ensure_ascii=False)
    except OSError as e:
        print(f"Warning: Could not save long-term memory: {e}")

In [ ]:
def update_memory(prompt: str, response: str):
    entry = {"prompt": prompt, "response": response}
    short_term_memory.append(entry)
    long_term_memory.append(entry)
    save_long_term_memory()

In [ ]:
def _simple_score(query: str, text: str) -> int:
    q_words = set(re.findall(r"\w+", query.lower()))
    t_words = set(re.findall(r"\w+", text.lower()))
    return len(q_words.intersection(t_words))

In [ ]:
def memory_context(query: str, short_items: int = 3, long_items: int = 5) -> str:
    recent_short = list(short_term_memory)[-short_items:]

    scored = []
    for item in long_term_memory:
        combined = f"{item.get('prompt', '')} {item.get('response', '')}"
        scored.append((_simple_score(query, combined), item))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_long = [item for score, item in scored[:long_items] if score > 0]

    if not top_long:
        top_long = list(long_term_memory)[-long_items:]

    lines = []
    if recent_short:
        lines.append("Recent short-term memory:")
        for idx, item in enumerate(recent_short, 1):
            lines.append(f"{idx}. User: {item['prompt']}")
            lines.append(f"   Assistant: {item['response']}")
    else:
        lines.append("Recent short-term memory: none.")

    if top_long:
        lines.append("")
        lines.append("Relevant long-term memory:")
        for idx, item in enumerate(top_long, 1):
            lines.append(f"{idx}. User: {item['prompt']}")
            lines.append(f"   Assistant: {item['response']}")
    else:
        lines.append("")
        lines.append("Relevant long-term memory: none.")

    return "\n".join(lines)

In [ ]:
def _recent_dialog_messages(turns: int = 4) -> list[dict[str, str]]:
    items = list(short_term_memory)[-turns:]
    messages: list[dict[str, str]] = []
    for item in items:
        prompt = item.get("prompt", "")
        response = item.get("response", "")
        if prompt:
            messages.append({"role": "user", "content": prompt})
        if response:
            messages.append({"role": "assistant", "content": response})
    return messages

In [ ]:
def _dialogue_aware_query(
    user_input: str,
    memory_query: str | None = None,
    turns: int = 3,
    max_chars: int = 900,
    include_answers: bool = False,
) -> str:
    parts: list[str] = [f"Current question: {user_input.strip()}"]

    if memory_query and memory_query.strip() and memory_query.strip() != user_input.strip():
        parts.append(f"Original user wording: {memory_query.strip()}")

    recent_items = list(short_term_memory)[-turns:]
    if recent_items:
        parts.append("Recent dialogue context:")
        for idx, item in enumerate(recent_items, 1):
            prompt = item.get("prompt", "").strip()
            response = item.get("response", "").strip()
            if prompt:
                parts.append(f"{idx}. User: {prompt}")
            if include_answers and response:
                parts.append(f"   Assistant: {response}")

    query = "\n".join(parts).strip()
    if len(query) > max_chars:
        query = query[-max_chars:]
    return query

## Extract content from the PDF file

In [ ]:
def extract_pdf_text(path_to_pdf: Path) -> str:
    with open(path_to_pdf, "rb") as f:
        pdf_reader = PdfReader(f)
        page_count = len(pdf_reader.pages)

        text_fragments = []
        for page in pdf_reader.pages:
            page_text = page.extract_text() or ""
            text_fragments.append(page_text)

    full_text = "\n".join(text_fragments)
    char_count = len(full_text)

    print(f"There are {page_count} pages in total.")
    print(f"There are {char_count} characters in total.")

    return full_text

## Chunk the PDF content

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [ ]:
def split_into_sentences(text: str):
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"\'])', text)
    return [s.strip() for s in sentences if s.strip()]

In [ ]:
def split_text(text: str, chunk_size=1000, overlap=200):
    # Split by paragraphs (blank lines)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        # Split each paragraph into sentences
        sentences = split_into_sentences(paragraph)

        for sentence in sentences:
            if len(current_chunk) + len(sentence) + 1 <= chunk_size:
                current_chunk += (" " + sentence) if current_chunk else sentence
            else:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())

                # Overlap: keep last N chars without breaking words
                overlap_text = current_chunk[-overlap:]
                if " " in overlap_text:
                    overlap_text = overlap_text.split(" ", 1)[-1]

                current_chunk = (overlap_text + " " + sentence).strip()

        # Force paragraph boundary if chunk already has content
        if current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = ""

    if current_chunk:
        chunks.append(current_chunk.strip())

    # Filter very tiny chunks
    chunks = [c for c in chunks if len(c) > 100]

    return chunks

## Setup the Vector Database

In [ ]:
chroma_client = PersistentClient(path=PATH_TO_CHROMA_DB)

try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print("Collection deleted.")
except Exception:
    print("Collection did not exist, skipping deletion.")

chroma_collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=OpenAIEmbeddingFunction(
        api_key=OPENAI_API_KEY,
        model_name="text-embedding-3-large")
    )

In [ ]:
def add_pdf_to_chroma(
    file_path: Path,
    chroma_collection,
    chunk_size=1000,
    overlap=200,
    batch_size=200
):
    start_time = time.time()

    print("Extracting text...")
    text = extract_pdf_text(file_path)

    if not text.strip():
        print(f"No text extracted from {file_path}")
        return

    print("Cleaning text...")
    text = clean_text(text)

    print("Chunking text...")
    chunks = split_text(text, chunk_size=chunk_size, overlap=overlap)
    print(f"Total chunks: {len(chunks)}")

    if not chunks:
        print("No chunks created after splitting.")
        return

    # Unique per run to avoid duplicate-id errors on re-ingest
    run_id = uuid4().hex[:8]
    ids = [f"{file_path.stem}_{run_id}_{i}" for i in range(len(chunks))]
    metadatas = [{"source": file_path.name} for _ in chunks]

    total_batches = (len(chunks) - 1) // batch_size + 1
    print(f"Adding chunks to Chroma in batches of {batch_size}...")

    for i in range(0, len(chunks), batch_size):
        batch_no = i // batch_size + 1
        try:
            chroma_collection.add(
                documents=chunks[i:i + batch_size],
                ids=ids[i:i + batch_size],
                metadatas=metadatas[i:i + batch_size]
            )
            print(f"Inserted batch {batch_no} / {total_batches}")
        except Exception as e:
            print(f"Batch {batch_no} failed: {e}")

    total_time = time.time() - start_time
    print(f"Finished adding PDF to Chroma in {total_time:.2f} seconds")

## CAG

In [ ]:
cag_system_prompt = """
You are a knowledgeable assistant.
Use provided memory and retrieved evidence.

Rules:
- Answer the user's question based ONLY on the provided context from previous interactions.
- Be specific and concise.
- Do NOT invent any information or add knowledge.
"""

In [ ]:
def _build_cag_context(user_input: str, n_results: int = 5) -> str:
    results = chroma_collection.query(
        query_texts=[user_input],
        n_results=n_results,
        include=["documents"],
    )
    context_docs = results.get("documents", [[]])[0]
    return "\n=====\n".join(context_docs)

In [ ]:
def retrieve_with_cag(
    user_input: str,
    stream_handler: Callable[[str], None],
    verbose: bool = True,
    memory_query: str | None = None,
) -> str:
    retrieval_query = _dialogue_aware_query(
        user_input=user_input,
        memory_query=memory_query,
        turns=3,
        include_answers=False,
    )
    context = _build_cag_context(retrieval_query, n_results=5)
    mem = memory_context(retrieval_query)
    recent_dialog = _recent_dialog_messages(turns=4)

    messages = [
        {"role": "developer", "content": cag_system_prompt},
        {"role": "developer", "content": f"Use this memory when resolving references:\n{mem}"},
        *recent_dialog,
        {"role": "assistant", "content": f"Retrieved context:\n{context}"},
        {"role": "user", "content": user_input},
    ]

    if verbose:
        print(f"[DEBUG] CAG: Query='{user_input[:50]}...'")
        if retrieval_query.strip() != f"Current question: {user_input.strip()}":
            print("[DEBUG] CAG: Using dialogue-aware retrieval query.")

    answer = ""

    with openai_client.responses.stream(
        model="gpt-5-nano",
        input=messages,
    ) as stream:

        for event in stream:
            if event.type == "response.output_text.delta":
                text = event.delta
                answer += text
                stream_handler(text)

        response = stream.get_final_response()

    if verbose:
        print(f"[DEBUG] CAG Response ID: {response.id}")
        if hasattr(response, "usage") and response.usage:
            usage = response.usage
            cached = getattr(getattr(usage, 'input_tokens_details', {}), 'cached_tokens', 0)
            reasoning = getattr(getattr(usage, 'output_tokens_details', {}), 'reasoning_tokens', 0)
            print(f"\n[DEBUG] Input tokens: {usage.input_tokens} (cached: {cached}); "
                  f"Output tokens: {usage.output_tokens} (reasoning: {reasoning})")

    return answer.strip()

## RAG

In [ ]:
class SearchContentArgs(BaseModel):
    """
    Search relevant passages from ingested PDF chunks
    using semantic similarity to find phrases matching the user's intent.
    Use this tool when a user asks about the content of a pdf or recommendations.
    """

    query: str = Field(description="The query string to search in the ingested PDF chunks")
    n_results: int = Field(default=5, ge=1, le=10)

def search_content(query: str, n_results: int = 5) -> dict:
    res = chroma_collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas"],
    )

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0] if "metadatas" in res else [{} for _ in docs]

    matches = []
    for i, doc in enumerate(docs):
        source = metas[i].get("source") if i < len(metas) and metas[i] else None
        matches.append({"rank": i + 1, "source": source, "text": doc})

    return {"matches": matches}

In [ ]:
def _build_rag_tools():
    schema = SearchContentArgs.model_json_schema()

    return [
        {
            "type": "function",
            "name": "search_content",
            "description": schema["description"],
            "parameters": {
                "type": "object",
                "properties": schema["properties"],
                "required": [*schema["properties"].keys()],
                "additionalProperties": False,
            },
            "strict": True
        }
    ]

In [ ]:
def _execute_tool_call(name: str, args: dict, tool_handlers: dict) -> dict:
    if name not in tool_handlers:
        return {"error": f"Unknown tool: {name}"}
    try:
        return tool_handlers[name](args)
    except Exception as e:
        return {"error": f"Tool {name} failed: {str(e)}"}

In [ ]:
rag_system_prompt = """
You are a knowledgeable assistant.

The user asks questions about PDF content.

You have access to:
1. Tools to search PDF content
2. Memory from previous interactions
3. Recent conversation context

Rules:
- You may use any tool available in `tool_handlers` to help answer the question.
- You must always call the appropriate handler from `tool_handlers` to get information.
- You must base your answer ONLY on retrieved PDF excerpts, memory, and conversation context.
- Do NOT invent any information or use external knowledge outside the provided context.
- If the retrieved content is insufficient, explicitly say it is insufficient.
"""

In [ ]:
def retrieve_with_rag(
    user_input: str,
    stream_handler: Callable[[str], None],
    verbose: bool = True,
    max_tool_rounds: int = 6,
    memory_query: str | None = None,
) -> str:

    tools = _build_rag_tools()

    def _tool_search_handler(args: dict) -> dict:
        validated = SearchContentArgs(**args).model_dump()
        raw_query = validated["query"]
        n_results = validated["n_results"]

        dialogue_query = _dialogue_aware_query(
            user_input=raw_query,
            memory_query=memory_query or user_input,
            turns=3,
            include_answers=False,
        )

        return search_content(query=dialogue_query, n_results=n_results)

    tool_handlers = {
        "search_content": _tool_search_handler
    }

    effective_query = _dialogue_aware_query(
        user_input=user_input,
        memory_query=memory_query,
        turns=3,
        include_answers=False,
    )

    mem = memory_context(effective_query)
    recent_dialog = _recent_dialog_messages(turns=4)

    response = openai_client.responses.create(
        model="gpt-5-nano",
        input=[
            {"role": "developer", "content": rag_system_prompt},
            {"role": "developer", "content": f"Use this memory when resolving references:\n{mem}"},
            *recent_dialog,
            {"role": "user", "content": effective_query},
        ],
        tools=tools
    )

    if verbose:
        print(f"\n[DEBUG] RAG: Query='{user_input[:50]}...'")
        if effective_query.strip() != f"Current question: {user_input.strip()}":
            print(f"[DEBUG] RAG: Using dialogue-aware retrieval query.")
        print(f"[DEBUG] Initial response ID: {response.id}")
        if hasattr(response, "usage") and response.usage:
            usage = response.usage
            cached = getattr(getattr(usage, 'input_tokens_details', {}), 'cached_tokens', 0)
            reasoning = getattr(getattr(usage, 'output_tokens_details', {}), 'reasoning_tokens', 0)
            print(f"[DEBUG] Input tokens: {usage.input_tokens} (cached: {cached}); "
                  f"Output tokens: {usage.output_tokens} (reasoning: {reasoning})")

    rounds = 0
    collected_matches: list[dict] = []

    while rounds < max_tool_rounds:
        rounds += 1

        function_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        if verbose:
            print(f"\n[DEBUG] Round {rounds}: {len(function_calls)} tool call(s)")

        if not function_calls:
            break

        tool_outputs = []

        for idx, call in enumerate(function_calls, 1):
            name = call.name
            args = json.loads(call.arguments or "{}")

            if verbose:
                print(f"[DEBUG] Tool {idx}/{len(function_calls)}: {name}")

            result = _execute_tool_call(name, args, tool_handlers)

            if isinstance(result, dict):
                matches = result.get("matches", [])
                if isinstance(matches, list):
                    for match in matches:
                        if isinstance(match, dict):
                            collected_matches.append({**match, "tool": name})

            if verbose:
                result_summary = str(result)[:100] + ("..." if len(str(result)) > 100 else "")
                print(f"[DEBUG] Tool result: {result_summary}")

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result, ensure_ascii=False),
                }
            )

        response = openai_client.responses.create(
            model="gpt-5-nano",
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools
        )

        if verbose:
            print(f"[DEBUG] Round {rounds} response ID: {response.id}")
            if hasattr(response, "usage") and response.usage:
                usage = response.usage
                cached = getattr(getattr(usage, 'input_tokens_details', {}), 'cached_tokens', 0)
                reasoning = getattr(getattr(usage, 'output_tokens_details', {}), 'reasoning_tokens', 0)
                print(f"[DEBUG] Input tokens: {usage.input_tokens} (cached: {cached}); "
                      f"Output tokens: {usage.output_tokens} (reasoning: {reasoning})")

    # Deterministic fallback: if the model does not call tools, force at least one retrieval pass.
    if not collected_matches:
        fallback_queries = [effective_query, user_input]
        if memory_query and memory_query.strip():
            fallback_queries.append(memory_query)

        seen_queries: set[str] = set()
        for q in fallback_queries:
            query = q.strip()
            if not query or query in seen_queries:
                continue
            seen_queries.add(query)

            dialogue_query = _dialogue_aware_query(
                user_input=query,
                memory_query=memory_query or user_input,
                turns=3,
                include_answers=False,
            )

            try:
                forced_result = search_content(query=dialogue_query, n_results=5)
                matches = forced_result.get("matches", []) if isinstance(forced_result, dict) else []
                if isinstance(matches, list) and matches:
                    for match in matches:
                        if isinstance(match, dict):
                            collected_matches.append({**match, "tool": "forced_search_content"})
                    if verbose:
                        print(f"[DEBUG] RAG fallback: forced retrieval succeeded for query='{query[:50]}...'")
                    break
                elif verbose:
                    print(f"[DEBUG] RAG fallback: no matches for query='{query[:50]}...'")
            except Exception as e:
                if verbose:
                    print(f"[DEBUG] RAG fallback error for query='{query[:50]}...': {e}")

    if not collected_matches and verbose:
        print("[DEBUG] No matches collected from tool outputs.")

    if not collected_matches:
        fallback = "I cannot answer from the uploaded PDF because no relevant excerpts were retrieved."
        stream_handler(fallback)
        return fallback

    excerpt_lines = []
    for i, m in enumerate(collected_matches[:8], 1):
        rank = m.get("rank", i)
        text = m.get("text", "")
        source = m.get("source", "unknown")
        excerpt_lines.append(f"[{rank}] ({source}) {text}")

    excerpts_block = "\n\n".join(excerpt_lines)

    if verbose:
        print()

    answer = ""
    with openai_client.responses.stream(
        model="gpt-5-nano",
        input=[
            {
                "role": "developer",
                "content": (
                    "Answer using the retrieved PDF excerpts, memory, and conversation context. "
                    "Cite supporting excerpts inline like [1], [2]. "
                    "Do NOT use external knowledge. "
                    "If the excerpts are insufficient, say so explicitly."
                ),
            },
            {"role": "developer", "content": f"Memory context:\n{mem}"},
            *recent_dialog,
            {"role": "assistant", "content": f"Retrieved excerpts:\n{excerpts_block}"},
            {"role": "user", "content": user_input},
        ],
    ) as stream:
        for event in stream:
            if event.type == "response.output_text.delta":
                text = event.delta
                answer += text
                stream_handler(text)

        final_response = stream.get_final_response()

    if verbose:
        print(f"\n\n[DEBUG] Final response ID: {final_response.id}")
        if hasattr(final_response, "usage") and final_response.usage:
            usage = final_response.usage
            cached = getattr(getattr(usage, 'input_tokens_details', {}), 'cached_tokens', 0)
            reasoning = getattr(getattr(usage, 'output_tokens_details', {}), 'reasoning_tokens', 0)
            print(f"[DEBUG] Input tokens: {usage.input_tokens} (cached: {cached}); "
                  f"Output tokens: {usage.output_tokens} (reasoning: {reasoning})")

    return answer.strip()

## Retrieve information (CAG or RAG)

In [ ]:
def retrieve_information(
    user_input: str,
    strategy: Literal["cag", "rag"],
    stream_handler: Callable[[str], None],
    verbose: bool = True,
    max_answer_iterations: int = 2,
    memory_query: str | None = None,
) -> str:
    for attempt in range(1, max_answer_iterations + 1):
        if strategy == "cag":
            answer = retrieve_with_cag(
                user_input,
                stream_handler=stream_handler,
                verbose=verbose,
                memory_query=memory_query,
            )
        elif strategy == "rag":
            answer = retrieve_with_rag(
                user_input,
                stream_handler=stream_handler,
                verbose=verbose,
                max_tool_rounds=6,
                memory_query=memory_query,
            )
        else:
            raise RuntimeError("Invalid strategy")

        if answer:
            return answer

        if verbose:
            print(f"[retry] Empty answer at attempt {attempt}/{max_answer_iterations}")

    return (
        "I could not generate an answer right now after multiple attempts. "
        "Please try rephrasing your question."
    )

## AI Query & Output Generation

In [ ]:
def _generate_image(user_input: str, answer: str) -> dict[str, str | None]:
    try:
        img = openai_client.images.generate(
            model="gpt-image-1-mini",
            prompt=(
                f'Question: "{user_input}"\n'
                f'Answer context: "{answer}"\n'
                "Create a photorealistic image based only on this context."
            ),
            size="1024x1024",
            quality="low",
            output_format="png",
        )
        image_b64 = img.data[0].b64_json if img.data else None
        if image_b64:
            display(HTML(f"<div style='white-space: pre-wrap; margin-bottom: 10px;'>{answer}</div>"))
            display(Image(data=base64.b64decode(image_b64)))
            return {
                "a": answer,
                "format": "image",
                "media": image_b64,
            }
    except Exception as e:
        print(f"Error generating image: {e}")

    return {"a": answer, "format": "text", "media": None}

In [ ]:
def _prepare_tts_script(user_input: str, answer: str) -> str:
    try:
        resp = openai_client.responses.create(
            model="gpt-5-nano",
            input=[
                {
                    "role": "developer",
                    "content": (
                        "Rewrite the provided answer into a plain spoken narration script. "
                        "If the user asked for a duration (for example, 10 seconds), try to match it. "
                        "If no duration is requested, do not force any specific length. "
                        "Do not add SSML, TTS tips, platform notes, options, or meta commentary. "
                        "No headings or bullet points; output one concise paragraph."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Original user request: {user_input}\n\n"
                        f"Source answer to narrate:\n{answer}"
                    ),
                },
            ],
        )
        script = (resp.output_text or "").strip()
        return script if script else answer
    except Exception as e:
        print(f"Warning: Could not prepare TTS script: {e}")
        return answer

def _generate_speech(
    user_input: str,
    answer: str,
    out_path: Path | None = None,
) -> dict[str, str | None]:
    tts_script = _prepare_tts_script(user_input=user_input, answer=answer)

    try:
        speech = openai_client.audio.speech.create(
            input=tts_script,
            model="gpt-4o-mini-tts",
            voice="alloy",
        )
        out_path = out_path or Path("/content/last_chefbot_answer.mp3")
        speech.write_to_file(str(out_path))
        with open(out_path, "rb") as f:
            audio_base64 = base64.b64encode(f.read()).decode()

        display(Audio(data=base64.b64decode(audio_base64), autoplay=False))
        return {
            "a": tts_script,
            "format": "speech",
            "media": audio_base64,
        }
    except Exception as e:
        print(f"Error generating speech: {e}")
        return {"a": answer, "format": "text", "media": None}

In [ ]:
def default_stream_printer(text: str) -> None:
    print(text, end="", flush=True)
    sys.stdout.flush()

In [ ]:
class IntermediateResponse(BaseModel):
    question: str
    expected_format: Literal["text", "image", "speech"]

def ask_ai(
    user_input: str,
    stream_handler: Callable[[str], None],
    strategy: Literal["cag", "rag"] = "rag",
    verbose: bool = True,
):
    parsed = openai_client.responses.parse(
        model="gpt-5-nano",
        input=[
            {
                "role": "developer",
                "content": (
                    "Extract the user's core question and desired response format. "
                    "Format must be text, image, or speech. "
                    "If uncertain, default to text."
                ),
            },
            {"role": "user", "content": user_input},
        ],
        text_format=IntermediateResponse,
    ).output_parsed

    if verbose:
        print("[DEBUG] Intermediate:", parsed)

    # Suppress text streaming in image mode; UI will show the generated image instead.
    effective_stream_handler = stream_handler
    if parsed.expected_format == "image":
        effective_stream_handler = lambda _: None

    answer = retrieve_information(
        parsed.question,
        strategy=strategy,
        stream_handler=effective_stream_handler,
        verbose=verbose,
        memory_query=user_input,
    )

    update_memory(user_input, answer)

    chat_entry = {"q": user_input, "a": answer, "format": "text", "media": None}

    if parsed.expected_format == "image":
        chat_entry.update(_generate_image(user_input=user_input, answer=answer))
    elif parsed.expected_format == "speech":
        chat_entry.update(_generate_speech(user_input=user_input, answer=answer))

    return chat_entry

## Interactive Chat UI

In [ ]:
chat_history: list[dict] = []

input_box = widgets.Text(
    placeholder='Ask a question...',
    layout=widgets.Layout(width='70%')
)

btn = widgets.Button(
    description='Ask AI',
    button_style='primary'
)

output_area = widgets.Output()

input_row = widgets.HBox([input_box, btn])

def render_chat(current_stream: str = "") -> None:
    html = ""

    for item in chat_history:
        html += f"""
        <div style="margin-bottom:15px;">
            <div><b>🧑 Question:</b> {item['q']}</div>
            <div style="
                margin-top:5px;
                padding:10px;
                border-radius:8px;
                background:#f1f1f1;
            ">
                <b>🤖 Answer:</b>
                <div style="white-space: pre-wrap; margin-bottom:10px;">
                    {item['a']}
                </div>
        """

        # Display media if available (image or speech)
        if item['format'] == 'image' and item['media']:
            html += f"""
                <div style="margin-top:10px;">
                    <b>📸 Image:</b>
                    <div>
                        <img src="data:image/png;base64,{item['media']}" style="max-width:512px; border-radius:8px;">
                    </div>
                </div>
            """
        elif item['format'] == 'speech' and item['media']:
            html += f"""
                <div style="margin-top:10px;">
                    <b>🔊 Audio:</b>
                    <div>
                        <audio controls style="width:100%;">
                            <source src="data:audio/mpeg;base64,{item['media']}" type="audio/mpeg">
                            Your browser does not support the audio element.
                        </audio>
                    </div>
                </div>
            """

        html += f"""
            </div>
        </div>
        """

    # Add streaming answer (last message in progress)
    if current_stream:
        html += f"""
        <div style="margin-bottom:15px;">
            <div><b>🤖 Answer:</b></div>
            <div style="
                padding:10px;
                border-radius:8px;
                background:#e8f4ff;
                font-family: monospace;
                white-space: pre-wrap;
            ">
                {current_stream}<span style="opacity:0.5;">▌</span>
            </div>
        </div>
        """

    # Wrap all messages in a scrollable div
    scrollable_html = f"""
    <div id="chefbot-chat-scroll" style="max-height:400px; overflow-y:auto;">
        {html}
    </div>
    """

    with output_area:
        output_area.clear_output(wait=True)
        display(HTML(scrollable_html))
        display(Javascript("""
        (function() {
            function scrollToBottom() {
                const target = document.getElementById('chefbot-chat-scroll');
                if (target) {
                    target.scrollTop = target.scrollHeight;
                }
            }
            scrollToBottom();
            requestAnimationFrame(scrollToBottom);
            setTimeout(scrollToBottom, 30);
            setTimeout(scrollToBottom, 120);
        })();
        """))


def make_stream_handler() -> tuple[Callable[[str], None], dict[str, str]]:
    streamed_text: dict[str, str] = {"value": ""}

    def ui_stream_handler(chunk: str) -> None:
        streamed_text["value"] += chunk
        render_chat(current_stream=streamed_text["value"])

    return ui_stream_handler, streamed_text


def on_button_click(_: widgets.Button) -> None:
    q = input_box.value.strip()
    if not q:
        return

    input_box.value = ""

    chat_history.append({"q": q, "a": "", "format": "text", "media": None})
    render_chat()

    ui_stream_handler, streamed_text = make_stream_handler()

    chat_entry = ask_ai(
        q,
        strategy="rag",
        verbose=False,
        stream_handler=ui_stream_handler,
    )

    chat_history[-1].update(chat_entry)

    render_chat()

## Run

#### Load Memory and Add Pdf to Chroma

In [ ]:
load_long_term_memory()

add_pdf_to_chroma(PDF_FILE, chroma_collection, chunk_size=1000, overlap=200, batch_size=200)

#### Test cases

In [ ]:
ask_ai("Could you suggest what I should cook for dinner?", strategy="rag", stream_handler=default_stream_printer, verbose=False);

In [ ]:
ask_ai("Give me more details about the first suggested option.", strategy="rag", stream_handler=default_stream_printer, verbose=True);

In [ ]:
ask_ai("Generate an image of that dish.", strategy="rag", stream_handler=default_stream_printer, verbose=False);

In [ ]:
ask_ai("I have potatoes, milk and eggs, what base recipe could I cook with this products?", strategy="rag", stream_handler=default_stream_printer, verbose=False);

In [ ]:
ask_ai("Please play the second recipe to be able to listen it. I want no more than 10 seconds record.", strategy="rag", stream_handler=default_stream_printer, verbose=True);

In [ ]:
ask_ai("Tell me what happened on March 3, 1878 in Bulgarian history?", strategy="rag", stream_handler=default_stream_printer, verbose=True);

In [ ]:
ask_ai("I want to prepare something sweet for dessert.", strategy="cag", stream_handler=default_stream_printer, verbose=True);

In [ ]:
ask_ai("Generate an image of one of the proposals above to show me how the dessert looks like.", strategy="cag", stream_handler=default_stream_printer, verbose=True);

In [ ]:
ask_ai("Generate a short audio (less than 15 seconds) with a SHOULD HE MARRY satirical jokes.", strategy="cag", stream_handler=default_stream_printer, verbose=False);

#### Input box

In [ ]:
btn.on_click(on_button_click, remove=True)
btn.on_click(on_button_click)
display(output_area, input_row)